# Rapport: Schemavalidering & Avvikelsedetektering av Sensor Data med Validation Pipeline
**Författare:** Marcus Bäckström  
**Kurs:** Valfri Fördjupning Python

## 1. Syfte 
--------

Syftet med projektet är att undersöka och demonstrera hur inkommande sensordata kan valideras och kvalitetssäkras utan att systemet gör osäkra gissningar.

Projektet visar hur lågriskfel och kända mönster — exempelvis teckenkodningsfel där "Örebro" blir "Orebro" vid dataöverföring — kan korrigeras automatiskt och deterministiskt. Samtidigt säkerställer pipelinen att mer komplexa eller orimliga avvikelser inte ändras godtyckligt. Genom att isolera osäkra rader för manuell granskning (Human-in-the-Loop) förhindras att felaktiga antaganden förvränger framtida analyser.

## 2. Området och dess relevans
----


Datavalidering är en grundpelare inom Data Science. Oavsett om syftet är att utföra explorativ dataanalys, bygga visualiseringar eller träna komplexa maskininlärningsmodeller, är alla slutresultat helt beroende av indatans kvalitet. Detta sammanfattas i branschprincipen "Garbage in, garbage out".

Om felaktiga antaganden görs eller om oren data släpps igenom i början av en pipeline uppstår felpropagation — små avvikelser i rådatan förstärks genom flödet och leder till fundamentalt felaktiga beslutsunderlag i slutändan. Som Data Scientist är det därför kritiskt att hålla en hög nivå av transparens och spårbarhet: det måste alltid gå att påvisa exakt vad datan innehöll från början och vilka transformationer som har utförts.

## 3. Viktiga begrepp
-----

* **Pydantic & Schemavalidering:** Pydantic är ett Python-bibliotek för datavalidering som använder datamodeller för att framtvinga typsäkerhet och struktur. Kärnan i Pydantic är klassen `BaseModel`, som definierar schemat och sätter de exakta reglerna för vilka datatyper, fält och gränsvärden indatan måste uppfylla.
* **Human-in-the-Loop (HITL):** Ett arkitekturval där automatisering underlättar databearbetning utan att ersätta mänskliga beslut. Istället för att koden godtyckligt raderar eller ändrar osäkra värden, städar systemet kända fel och flaggar resten så att en människa enkelt kan granska avvikelserna.
* **Spårbarhet (Data Lineage):** Förmågan att spåra data från slutprodukten hela vägen tillbaka till ursprungskällan. När automatisk städning utförs sparas originaldatan (t.ex. i `original_region`) så att det alltid går att auditera vad datan innehöll från början.
* **Deterministisk datastädning:** Förbestämd och regelbaserad datatvätt där korrigeringar utförs med full säkerhet. Metoden rättar förutsägbara mönsterfel utan att introducera gissningar som kan försämra datakvaliteten.

 ## 4. Genomförande
 -----

### Verktyg & Data
* **Stack:** Python 3.13, Pandas (datastädning & tabeller), Pydantic (schemavalidering) och Pythons inbyggda `logging`.
* **Testdata:** 5 syntetiska JSON-filer genererade i `src/generate_data.py` (korrekt data, null-värden, stavfel, outliers samt ett kombinerat dataset).

### Arkitektur & Flöde
Koden är modulär och uppdelad under `src/` (`io.py`, `transform.py`, `validate.py`, `log_config.py`).
1. **Pandas** läser in JSON-data och utför en snabb, deterministisk städning av kända teckenfel ("Orebro" -> "Örebro").
2. **Pydantic** kör varje rad mot schemat `TemperatureRead`. Istället för att kasta bort trasiga rader sätter pipelinen `flagged_for_manual_review = True` och skickar en `WARNING` till loggfilen för Human-in-the-Loop-granskning.

### Huvudsakligt problem & Lösning
* **Problem:** Pydantics `BaseModel` validerar enskilda dictionaries, medan projektet använde Pandas DataFrames för tabellvisning och städning.
* **Lösning:** DataFramen konverteras tillfälligt till en lista av dicts (`.to_dict('records')`) inne i `validate.py` enbart under valideringssteget, för att sedan återbyggas till en uppdaterad DataFrame med flaggkolumner.

In [ ]:
from src.log_config import configure_logging
from src.generate_data import generate_mock_data, MOCK_DATA_FOLDER, DATA_NAMES
from src.io import load_json_to_dataframe, save_csv
from src.transform import auto_correct_regions
from src.validate import validate_temperature_records

#Configure logger
configure_logging()

#Generate 5 mock data with different problems.
generate_mock_data()

Saved 5 mock data json files to data/


In [5]:
# Läs in mixed_failure_data.json och skapa DataFrame
df_raw = load_json_to_dataframe(f"{MOCK_DATA_FOLDER}/mixed_failure_data.json") 
df_raw

2026-09-18 14:30:55 | INFO | temperature_log | Successfully read JSON file at: data\mixed_failure_data.json


,sensor_id,region,sector,temperature
0,S1,Örebro,West,20.0
1,S2,Orebro,C,51.2
2,S3,orobo,SE,20.3
3,S4,Orebro,Söder,30.5
4,S5,sthlm,C,10.0


## 5. Resultat 
-----

Nedan demonstreras pipelinen steg för steg på datasetet `mixed_failure_data.json` för att visa hur oren rådata bearbetas, städas och valideras.

### Exekveringssteg:
1. **Steg 1 (Rådata):** Funktionen `load_json_to_dataframe()` läser in en JSON-fil från mappen `data/`. Detta simulerar inkommande data från externa sensorer och konverterar strukturen till en Pandas DataFrame för lokal hantering och visualisering.
2. **Steg 2 (Pandas-städning):** DataFramen skickas till `auto_correct_regions()`. Funktionen städar förutsägbara småfel (som "sthlm" eller "orebro"). Som standard används den oföränderliga konfigurationen `KNOWN_REGION_MAPPINGS (MappingProxyType)`, men anpassade dictionaries godtas också. Genom `.strip().lower()` görs sökningen oberoende av mellanslag och teckenstorlek. På de rader som korrigeras sparas ursprungsvärdet i `original_region` för spårbarhet och `auto_cleaned` sätts till `True`. Händelsen loggas därefter via `logger.info`.
3. **Steg 3 (Pydantic-validering):** Den tvättade DataFramen skickas vidare till `validate_temperature_records()`, som även tar emot `dataset_name` för spårbar loggning. Funktionen validerar posterna mot schemat `TemperatureRead` (en Pydantic `BaseModel` i `schemas.py`). Schemat tillämpar strikt typhantering via `Literal` för sektorer, gränsvärdeskontroll (-40°C till +40°C) samt en anpassad `@field_validator` som filtrerar bort placeholder-strängar ("null", "n/a") och kräver att regionen finns i `ALLOWED_REGIONS`. Eftersom Pydantic validerar enskilda objekt konverteras DataFramen tillfälligt via `.to_dict(orient="records")`. Om en rad bryter mot schemat fångas `ValidationError`, den booleska kolumnen `flagged_for_manual_review` sätts till `True`, och en `WARNING`-logg genereras med antalet flaggade rader.

In [ ]:
#städar common misspelling eller förkortningar. använder antingen KNOWN_REGION_MAPPING om mapping i functioned är none,
#men det gör att göra en egen dict.
df_cleaned = auto_correct_regions(df_raw)
df_cleaned

2026-09-18 14:55:16 | INFO | temperature_log | Auto-corrected 3 region entries.


,sensor_id,region,sector,temperature,original_region,auto_cleaned
0,S1,Örebro,West,20.0,None,False
1,S2,Örebro,C,51.2,Orebro,True
2,S3,orobo,SE,20.3,None,False
3,S4,Örebro,Söder,30.5,Orebro,True
4,S5,Stockholm,C,10.0,sthlm,True


In [ ]:
#Validerar den städade datafarmen emot Pydantic schema som gjort´s i schemas.py  
df_validate = validate_temperature_records(df_cleaned, dataset_name="mixed_failure_data")
df_validate

2026-09-18 14:53:48 | WARNING | temperature_log | in mixed_failure_data: 3 of 5 rows were flagged for manual review.


,sensor_id,region,sector,temperature,original_region,auto_cleaned,flagged_for_manual_review
0,S1,Örebro,West,20.0,None,False,False
1,S2,Örebro,C,51.2,Orebro,True,True
2,S3,orobo,SE,20.3,None,False,True
3,S4,Örebro,Söder,30.5,Orebro,True,True
4,S5,Stockholm,C,10.0,sthlm,True,False


## 6. Begränsningar och möjliga förbättringar
------

### Identifierade begränsningar
* **Enkel regelvalidering utan kontext (Rumslig avvikelse):** Schemat i Pydantic kontrollerar enbart statiska gränsvärden (-40°C till +40°C). Systemet saknar logik för att jämföra mätningar mellan närliggande sektorer. En mätning på 30°C i Örebro Syd samtidigt som Örebro Sydost visar 20°C släpps igenom, trots att en skillnad på 10°C i samma område tyder på mätfel.
* **Saknad tidsdimension och dubbletthantering:** Schemat saknar tidsstämplar (`timestamp`). Det går därför inte att validera om temperaturer ändras orimligt snabbt över tid, eller om samma sektor råkat skicka flera mätningar vid samma tidpunkt.
* **Beroende av exakt nyckeluppslagning:** Även om `auto_correct_regions()` är flexibel och tar emot anpassade dictionaries via `mapping`-argumentet, bygger städningen på exakt matchning (`exact match`). Om en felstavning inte finns med i mappen lämnas den orörd.
* **Formatkoppling:** Pipelinen läser i dagsläget enbart JSON-filer från disk och kräver konvertering till Pandas DataFrames i samtliga steg.

### Möjliga förbättringar
* **Rumslig och statistisk validering:** Införa grannskapsjämförelser eller rullande medelvärden för att flagga orimliga lokala avvikelser mellan sektorer.
* **Tidsstämplar och aggregering:** Utöka `TemperatureRead`-schemat med ett obligatoriskt `timestamp`-fält för tidsserievalidering och rensning av dubbletter per tidsintervall.
* **Fuzzy Matching (Algoritmitisk tvätt):** Komplettera ordboksuppslagningen med bibliotek som *RapidFuzz* för att automatiskt fånga upp och rätta okända felstavningar baserat på stränglikhet.
* **Format-agnostisk ingestion:** Utöka `src/io.py` för att stödja CSV, Parquet samt direktinläsning från API-strömmar.

## 7. Koppling till yrkesrollen
-----

I rollen som Data Scientist, Data Engineer eller dataanalytiker är hantering av oren och ostrukturerad indata en av de mest tidskrävande delarna av arbetet. Oavsett domän är strukturerad validering ett absolut krav för att garantera att datan faktiskt går att lita på i efterföljande analyser och modeller.

### Tillämpning i pipelines och produktion
* **Mönster för datakontrakt (Data Contracts):** När ett system tar emot data från tiotals olika källor eller sensorer fungerar ett Pydantic-schema som en central mall. Genom att validera datan direkt vid inläsningsskiktet (*ingestion layer*) stoppas trasig data från att läcka in och förstöra downstream-pipelines.
* **Gränsdragning mellan städning och flaggning:** En central del i yrkesrollen är att sätta strikta regler för vad systemet får ändra automatiskt kontra vad det bara ska flagga. Genom att begränsa automatisk tvätt till säkra mönster och flagga osäkra värden skyddas dataintegriteten samtidigt som manuella granskare (Human-in-the-Loop) får ett effektivt underlag.
* **Förebyggande av fel i maskininlärning (MLOps):** För en Data Scientist innebär trasig indata att maskininlärningsmodeller generera felaktiga prediktioner. Automatisk schemavalidering fungerar som en skyddsmekanism som upptäcker dataavvikelser (*data drift* eller felaktiga typer) innan modellerna tränas eller utvärderas på felaktiga grundvalar.

## 8. Källor 
----

- Pydantic (Data validering): https://pydantic.dev/docs/
- Pandas (Data hantering): https://pandas.pydata.org/docs/
---
- AI https://gemini.google.com/ för frågor och bolla ideer och få bättre struktur och hitta lösningar till problem.

## 9. Självreflektion 
-----

### 1. Vad lärde du dig som du inte kunde innan?
Jag lärde mig hur Pydantic-scheman struktureras i praktiken och hur man överbryggar friktionen när ett bibliotek kräver dictionaries (`list[dict]`) medan resten av pipelinen använder Pandas DataFrames. Jag fick även en djupare förståelse för pipeline-arkitektur — att våga refaktorera och ändra innandömet i en funktion utan att bryta dess interface. Slutligen lärde jag mig navigera i Pydantics dokumentation, som ofta är inriktad mot webbdata (t.ex. e-postadresser och heltal) snarare än numeriska intervallkontroller för data science.

### 2. Vad var svårast att förstå eller genomföra?
Det svåraste var att avgöra exakt *var*, *när* och *hur* valideringen skulle genomföras i dataflödet. Det var en tröskel att kombinera Pandas och Pydantic genom att konvertera strukturen tillfälligt. Även felhanteringen var utmanande: att bygga en `try-except ValidationError`-loop som inte stoppar koden vid fel, utan istället kontrollerat fångar avvikelser och sätter en boolesk flagga.

### 3. Vilket tekniskt val är du mest nöjd med och varför?
Att jag tidigt beslutade att kombinera automatisk städning med strikt spårbarhet och schemavalidering. Pipelinen rättar först kända mönsterfel (som "Orebro" till "Örebro") utan att förstöra originaldata (via `original_region`), för att därefter köra en hård Pydantic-validering mot `ALLOWED_REGIONS` och gränsvärden. Detta tvåstegsförfarande gör datatvätten både säker och spårbar.

### 4. Vad hade du gjort annorlunda om du började om?
Jag hade byggt projektet i en mer logisk ordning från start: `log_config` -> `mock_data` -> `schemas` -> `transform/correct spelling` -> `validate`. Nu skapades schemat sent i processen, vilket krävde omstrukturering av tidigare moduler. Jag hade även inkluderat en `timestamp`-kolumn i schemat från början samt sparat det faktiska felmeddelandet från Pydantic i en ny kolumn istället för att enbart sätta en boolesk flagga.

### 5. Vad skulle vara ett naturligt nästa steg om du fortsatte arbetet?
1. Införa rumslig och statistisk validering som jämför temperaturer mellan närliggande sektorer i samma region för att upptäcka orimliga lokala avvikelser.
2. Utöka DataFramen med en kolumn för felorsak (`validation_error`) som sparar Pydantics exakta felmeddelande för snabbare manuell felsökning.

### 6. Betygssättning
**Självskattat betyg:** G (alternativt VG baserat på den modulariserade arkitekturen).

### 7. Motivering till betyg
* **Val och avgränsning:** Problemet är direkt kopplat till yrkesrollen som Data Scientist. Datavalidering och ingestion-kontroll är kritiska moment i alla datapipelines för att förhindra att felaktig data når analysmodeller.
* **Genomförande:** Pipelinen är uppbyggd kring ett Pydantic-schema (`TemperatureRead`) som använder `BaseModel`, `Literal` för sektorer, `Field`-restriktioner (`ge=-40.0, le=40.0`) samt en anpassad `@field_validator` för placeholder-rensning och verifiering mot `ALLOWED_REGIONS`. Datagränssnittet mellan Pandas och Pydantic hanteras korrekt via kontrollerad typkonvertering och undantagshantering (`try-except ValidationError`).
* **Kod och dokumentation:** Projektet har en ren och modulär struktur under `src/` med separerade ansvarsområden (`io`, `transform`, `validate`, `log_config`). `README.md` ger tydliga instruktioner för exekvering, och koden innehåller loggning på rätt nivåer (`INFO`/`WARNING`) för full audit-spårbarhet.